# Calidad de datos — Baches de Hermosillo

Este notebook analiza la calidad del dataset enriquecido de baches antes de decidir qué reglas y columnas deben conservarse en el pipeline final.

## Objetivos
- Revisar completitud y valores faltantes.
- Detectar duplicados y validar identificadores.
- Validar fechas y consistencia entre `date`, `date_mx` y `year`.
- Revisar coordenadas y posibles valores atípicos espaciales.
- Analizar la calidad del enriquecimiento de colonia.
- Revisar `colonia_match`, `spatial_match_method` y `distance_to_colonia_m`.
- Detectar columnas redundantes o exclusivamente de auditoría.
- Proponer un conjunto de columnas finales para el dataset tidy.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)


## 1. Cargar dataset


In [ ]:
DATA_PATH = Path("../data/processed/bachometro/baches_hermosillo_colonia_filled.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())


In [ ]:
df.info()


## 2. Perfil general de calidad


In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "nulos_pct": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique(dropna=True),
})

quality = quality.sort_values(["nulos_pct", "unicos"], ascending=[False, False])
display(quality)


### Interpretación inicial
- Los nulos en `colonia_original` son esperables porque muchos reportes no traían colonia desde la fuente original.
- `colonia_match` es nulo cuando no existen ambos valores para comparar.
- `settlement_name` puede quedar nulo si el punto no obtuvo una colonia espacial.
- `colonia` debería estar completamente poblada después del fallback final.


## 3. Duplicados e identificadores


In [ ]:
print("Filas duplicadas completas:", df.duplicated().sum())
print("id duplicados:", df["id"].duplicated().sum())
print("id_row duplicados:", df["id_row"].duplicated().sum())

print("\n¿id es único?:", df["id"].is_unique)
print("¿id_row es único?:", df["id_row"].is_unique)


In [ ]:
duplicated_ids = df[df["id"].duplicated(keep=False)].sort_values("id")
display(duplicated_ids.head(50))


## 4. Calidad de fechas


In [ ]:
df["date_parsed"] = pd.to_datetime(df["date"], errors="coerce")
df["date_mx_parsed"] = pd.to_datetime(
    df["date_mx"],
    format="%d/%m/%Y",
    errors="coerce",
)

print("date inválidas:", df["date_parsed"].isna().sum())
print("date_mx inválidas:", df["date_mx_parsed"].isna().sum())


In [ ]:
date_mismatch = df[
    df["date_parsed"].notna()
    & df["date_mx_parsed"].notna()
    & (df["date_parsed"] != df["date_mx_parsed"])
]

print("date vs date_mx inconsistentes:", len(date_mismatch))
display(date_mismatch[["id", "date", "date_mx"]].head(30))


In [ ]:
year_mismatch = df[
    df["date_parsed"].notna()
    & (df["date_parsed"].dt.year != df["year"])
]

print("year inconsistente con date:", len(year_mismatch))
display(year_mismatch[["id", "date", "year"]].head(30))


Si `date` y `date_mx` contienen exactamente la misma información, conviene conservar solo `date` como fecha canónica y eliminar `date_mx` del dataset analítico.


## 5. Validación de coordenadas


In [ ]:
invalid_global_coords = df[
    ~df["latitude"].between(-90, 90)
    | ~df["longitude"].between(-180, 180)
]

print("Coordenadas globalmente inválidas:", len(invalid_global_coords))
display(invalid_global_coords[["id", "latitude", "longitude"]].head(30))


In [ ]:
display(
    df[["latitude", "longitude"]]
    .describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99])
)


## 6. Variables categóricas principales


In [ ]:
categorical_columns = [
    "material",
    "colonia_source",
    "spatial_match_method",
    "contexto_ubicacion",
    "fuera_asentamiento",
    "settlement_type",
]

for column in categorical_columns:
    if column in df.columns:
        print(f"\n--- {column} ---")
        display(df[column].value_counts(dropna=False).to_frame("count"))


## 7. Calidad del enriquecimiento espacial


In [ ]:
print("Método espacial:")
display(df["spatial_match_method"].value_counts(dropna=False).to_frame("count"))

print("\nFuente de colonia final:")
display(df["colonia_source"].value_counts(dropna=False).to_frame("count"))


In [ ]:
display(
    df["distance_to_colonia_m"]
    .describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
)


In [ ]:
display(
    df.groupby("spatial_match_method", dropna=False)["distance_to_colonia_m"]
    .describe()
)


In [ ]:
plt.figure(figsize=(10, 5))
df.loc[df["distance_to_colonia_m"] > 0, "distance_to_colonia_m"].hist(bins=60)
plt.xlabel("Distancia a colonia más cercana (m)")
plt.ylabel("Número de reportes")
plt.title("Distribución de distancia para puntos fuera de polígonos")
plt.show()


## 8. Comparación de colonia original vs colonia espacial


In [ ]:
display(
    df["colonia_match"]
    .value_counts(dropna=False)
    .to_frame("count")
)


In [ ]:
discrepancias = df[df["colonia_match"].eq(False)].copy()

print("Número de discrepancias:", len(discrepancias))

display(
    discrepancias[[
        "id",
        "latitude",
        "longitude",
        "colonia_original",
        "colonia_original_normalized",
        "settlement_name",
        "settlement_name_normalized",
        "distance_to_colonia_m",
        "spatial_match_method",
    ]].head(100)
)


In [ ]:
comparables = df["colonia_match"].notna().sum()
mismatches = df["colonia_match"].eq(False).sum()

if comparables:
    print("Porcentaje de discrepancias:", round(mismatches / comparables * 100, 2), "%")


## 9. Revisar la columna final `colonia`


In [ ]:
print("Nulos en colonia final:", df["colonia"].isna().sum())
print("Valores únicos:", df["colonia"].nunique())

display(df["colonia"].value_counts(dropna=False).head(50).to_frame("count"))


In [ ]:
context_values = [
    "cerca_asentamiento",
    "probable_carretera_o_zona_rural",
]

display(
    df[df["colonia"].isin(context_values)][[
        "id",
        "latitude",
        "longitude",
        "distance_to_colonia_m",
        "contexto_ubicacion",
        "colonia",
        "colonia_source",
    ]].head(100)
)


### Consideración de diseño
`colonia` mezcla actualmente nombres de asentamientos con categorías como `probable_carretera_o_zona_rural`.

Una alternativa más tidy sería:
- `colonia`: únicamente nombre de colonia/asentamiento o `NA`.
- `contexto_ubicacion`: `dentro_asentamiento`, `cerca_asentamiento`, `probable_carretera_o_zona_rural`.

Conviene decidirlo después de revisar los resultados.


## 10. Revisar redundancia de columnas


In [ ]:
redundancy_checks = {
    "date_vs_date_mx": (
        (df["date_parsed"] == df["date_mx_parsed"])
        | (df["date_parsed"].isna() & df["date_mx_parsed"].isna())
    ).mean(),
    "id_vs_id_row": (df["id"] == df["id_row"]).mean(),
}

display(pd.Series(redundancy_checks, name="proporcion_igual"))


Columnas candidatas a ser únicamente de auditoría:
- `colonia_original`
- `colonia_original_normalized`
- `settlement_name`
- `settlement_name_normalized`
- `colonia_match`
- `spatial_match_method`

No se recomienda eliminarlas hasta haber terminado la validación del enriquecimiento espacial.


## 11. Propuesta inicial de dataset final


In [ ]:
FINAL_COLUMNS = [
    "id",
    "latitude",
    "longitude",
    "date",
    "year",
    "material",
    "description",
    "neighborhoods",
    "colonia",
    "colonia_source",
    "settlement_type",
    "distance_to_colonia_m",
    "fuera_asentamiento",
    "contexto_ubicacion",
]

missing_final_columns = [col for col in FINAL_COLUMNS if col not in df.columns]
print("Columnas faltantes:", missing_final_columns)

df_final_candidate = df[
    [col for col in FINAL_COLUMNS if col in df.columns]
].copy()

display(df_final_candidate.head())
print("Shape candidato:", df_final_candidate.shape)


## 12. Validaciones candidatas para mover al job


In [ ]:
checks = {
    "id_sin_nulos": df["id"].notna().all(),
    "id_unico": df["id"].is_unique,
    "latitude_sin_nulos": df["latitude"].notna().all(),
    "longitude_sin_nulos": df["longitude"].notna().all(),
    "latitude_valida": df["latitude"].between(-90, 90).all(),
    "longitude_valida": df["longitude"].between(-180, 180).all(),
    "colonia_final_sin_nulos": df["colonia"].notna().all(),
    "distance_sin_nulos": df["distance_to_colonia_m"].notna().all(),
    "distance_no_negativa": df["distance_to_colonia_m"].ge(0).all(),
}

display(pd.Series(checks, name="cumple"))


## 13. Conclusiones y decisiones

Completa esta sección después de ejecutar el notebook:

- ¿`id` puede reemplazar a `id_row`?
- ¿`date_mx` es redundante respecto a `date`?
- ¿Los umbrales espaciales actuales son razonables?
- ¿Qué porcentaje de `colonia_match=False` existe entre registros comparables?
- ¿Se deben conservar las columnas normalizadas en el dataset final o únicamente en auditoría?
- ¿`colonia` debe contener categorías de contexto o únicamente nombres de asentamiento?
- ¿Qué columnas deben quedar en el CSV final del pipeline?
